#### Experimentation playground for translating Feature Engineering INTO SQL for `gold layer`

in the silver and gold could we implement zero copy integration by passing same dataframe from the cleaning to that.
or is the featuren engienering and join overhead more than that.

In [1]:
import duckdb
import config

In [2]:
con = duckdb.connect('../duckdb/master_db.duckdb')
con.execute('INSTALL delta; LOAD delta')

- create a table weekly_sales for original data

In [ ]:
con.sql(f"""
    CREATE TABLE IF NOT EXISTS weekly_sales AS 
    SELECT 
        *
    FROM delta_scan('{config.SILVER_SALES_PATH}') s
    LEFT JOIN delta_scan('{config.SILVER_FEATURES_PATH}') f
        ON s.store = f.store AND s.date = f.date
    LEFT JOIN delta_scan('{config.SILVER_STORES_PATH}') st
        ON s.store = st.store
    """
)

In [25]:
con.sql('SELECT COUNT(*) FROM weekly_sales LIMIT 5')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       421570 │
└──────────────┘

- create the continous scaffold from previous eda's

In [26]:
sql = """
    CREATE OR REPLACE TABLE scaffold_sales AS 
    WITH calender AS 
    -- Generate Continous weekly Dates
    (SELECT 
         * 
    FROM GENERATE_SERIES(
        (SELECT MIN(date) FROM weekly_sales),
        (SELECT MAX(date) FROM weekly_sales),
        INTERVAL 7 DAYS
    ) AS t(date)),

    -- Get unique store dept combinations
    unique_stores AS 
    (SELECT 
        DISTINCT store,
        dept
     FROM weekly_sales),

     -- Cross join to create the continous timeline
    scaffold AS (
    SELECT 
        u.store, u.dept, c.date
    FROM unique_stores u
    CROSS JOIN calender c ),

    merged_data AS (
    SELECT
        *
    FROM scaffold s
    LEFT JOIN weekly_sales AS sales 
        ON s.store = sales.store AND s.dept = sales.dept AND s.date = sales.date)
        
    SELECT * FROM merged_data;

"""

con.sql(sql)

In [27]:
con.sql(
    """
    SELECT COUNT(*) FROM scaffold_sales LIMIT 5
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       476333 │
└──────────────┘

- Logic for creating other features

In [29]:
con.sql(
    """
    CREATE OR REPLACE TABLE features AS  
    SELECT 
        store,
        dept,
        date,
        LN(weekly_sales + 1) AS sales_log,
        SIN(2 * PI() * WEEK(date)/52) AS sin_week,
        COS(2 * PI() * WEEK(date)/52) AS cos_week,
        LAG(LN(weekly_sales + 1), 1) OVER w_momentum AS lag_1,
        LAG(LN(weekly_sales + 1), 5) OVER w_momentum AS lag_5
    FROM scaffold_sales
    WINDOW w_momentum AS (PARTITION BY store, dept ORDER BY date);
    """

)

In [31]:
con.sql("""SELECT * FROM features""")

┌───────┬───────┬─────────────────────┬────────────────────┬─────────────────────┬─────────────────────────┬────────────────────┬────────────────────┐
│ store │ dept  │        date         │     sales_log      │      sin_week       │        cos_week         │       lag_1        │       lag_5        │
│ int32 │ int32 │      timestamp      │       double       │       double        │         double          │       double       │       double       │
├───────┼───────┼─────────────────────┼────────────────────┼─────────────────────┼─────────────────────────┼────────────────────┼────────────────────┤
│    39 │    41 │ 2010-08-06 00:00:00 │  6.309700072603219 │ -0.5680647467311556 │     -0.8229838658936565 │ 6.5638555265321274 │  5.522580291130163 │
│    39 │    41 │ 2010-08-13 00:00:00 │  7.622169700043039 │  -0.663122658240795 │     -0.7485107481711013 │  6.309700072603219 │  6.363924253602162 │
│    39 │    41 │ 2010-08-20 00:00:00 │  7.214607425769474 │ -0.7485107481711011 │     -0.6631